# Biogeochemical (BGC) analysis at CTD sites

This notebook compares observed and modeled DIC, alkalinity, and nutrients at CTD stations (HV stations) in Hvalfjörður.

- **Observations**: Load processed HAFRO BGC casts and convert to volumetric units for Alk and DIC.
- **Model fields**: Regrid Iceland2 BGC and physical fields to fixed depth levels and extract around HV stations.
- **Normalized metrics**: Compute salinity-normalized Alk/DIC in both data and model to separate freshwater vs internal processes.
- **Spatial structure**: Create along-fjord sections, boxplots by cruise, and depth profiles to visualize gradients.
- **Nutrient comparison**: Analyze NO₃, PO₄, SiO₃, and NH₄ distributions (surface and full water column) across time.

Use this notebook to assess whether the model reproduces the observed carbon and nutrient structure and variability.

In [2]:
import subprocess
import os
import pandas as pd
#import netCDF4
import numpy as np
import glob
import time
import matplotlib.pyplot as plt
import copy
import xarray as xr
from datetime import datetime, timedelta 
import dask
from scipy.interpolate import griddata
#from ocean_c_lab_tools import *
#from celluloid import Camera 
#import PyCO2SYS as csys
import gsw as sw
from roms_regrid import *

In [28]:
model_grid_path="/home/x-uheede/S/Iceland2_MARBL_2024_60m/P_INPUT/Iceland2_grid.nc"
obs_bgc_path='/anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data/staged/seanoe_hvalfjordur/2026-04-27/seanoe_110401/discrete_samples_qc/discrete_samples_qc.xlsx'

bgc_model_paths = [
    "/home/x-uheede/S/Iceland2_MARBL_2024_60m/Iceland2_MARBL_2024_bgc.20240?????????.nc"
]

phys_model_paths = [
    "/home/x-uheede/S/Iceland2_MARBL_2024_60m/Iceland2_MARBL_2024_his.20240?????????.nc"
]


In [20]:
xls = pd.ExcelFile(obs_bgc_path)
combo = pd.read_excel(xls, 'Sheet1', decimal='.', header=27, skiprows=[28])

# We use .str.replace with regex=False for a simple string swap
combo['HV'] = combo['Station_name'].str.replace('HV', '', regex=False).astype(int)

# 2. Create the 'time' column
combo['time'] = pd.to_datetime(combo[['Year_UTC', 'Month_UTC', 'Day_UTC']].rename(
    columns={'Year_UTC': 'year', 'Month_UTC': 'month', 'Day_UTC': 'day'}
))

# -----------------------------
# CREATE DEPTH BINS (0–9.9, 10–19.9, ...)
# -----------------------------
combo["depth_bin"] = (np.floor(combo["Depth"] / 10) * 10).astype(int)

obs=xr.Dataset.from_dataframe(combo)
obs=obs.set_index(index=['HV','depth_bin','time'])
obs=obs.drop_duplicates('index')
obs=obs.unstack('index')
obs=obs.rename(name_dict={'Latitude':'lat','Longitude':'lon'})

In [13]:
combo

,Exp_ID,Cruise_ID,Station_ID,Station_name,Year_UTC,Month_UTC,Day_UTC,Start_time_UTC,Latitude,Longitude,...,Ammonium_flag,chl_a,phaeo,Diatom,Dinoflagellate,Ciliate,Total,HV,time,depth_bin
0,46BS20240404,B5-2024,186,HV1,2024,4,4,08:56:00,64.2638,-21.9867,...,2,2.07,0.31,26.0,9.0,0.0,35.0,1,2024-04-04,0
1,46BS20240404,B5-2024,186,HV1,2024,4,4,08:56:00,64.2638,-21.9867,...,2,2.16,0.26,28.0,11.0,0.0,39.0,1,2024-04-04,10
2,46BS20240404,B5-2024,186,HV1,2024,4,4,08:56:00,64.2638,-21.9867,...,2,2.93,0.37,NaN,NaN,NaN,NaN,1,2024-04-04,30
3,46BS20240404,B5-2024,192,HV12,2024,4,4,13:25:00,64.3643,-21.4522,...,2,2.24,0.14,NaN,NaN,NaN,NaN,12,2024-04-04,0
4,46BS20240404,B5-2024,192,HV12,2024,4,4,13:25:00,64.3643,-21.4522,...,2,5.92,0.59,NaN,NaN,NaN,NaN,12,2024-04-04,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,46VO20250428,VO2-2025,23,HV10,2025,4,28,15:18:00,64.3820,-21.5267,...,2,3.93,1.29,7.0,0.0,5.0,12.0,10,2025-04-28,0
484,46VO20250428,VO2-2025,23,HV10,2025,4,28,15:18:00,64.3820,-21.5267,...,2,3.38,1.26,15.0,0.0,10.0,25.0,10,2025-04-28,10
485,46VO20250428,VO2-2025,23,HV10,2025,4,28,15:18:00,64.3820,-21.5267,...,2,3.29,2.05,NaN,NaN,NaN,NaN,10,2025-04-28,30
486,46VO20250428,VO2-2025,23,HV10,2025,4,28,15:18:00,64.3820,-21.5267,...,2,3.07,1.90,NaN,NaN,NaN,NaN,10,2025-04-28,60


In [21]:
density=sw.density.sigma0(obs['Salinity'],obs['Temperature_CTD'])
obs['Alk']=obs['TA'] * (1 / 1000) * (density+1000)
obs['DIC']=obs['DIC'] * (1 / 1000) * (density+1000)

/home/x-uheede/.local/lib/python3.11/site-packages/xarray/computation/apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sigma0
  result_data = func(*input_data)


In [22]:
TA0=282
Sref=34
DIC0=313
nTA=((obs['Alk']-TA0)/obs['Salinity'])*Sref+TA0

nDIC=((obs['DIC']-DIC0)/obs['Salinity'])*Sref+DIC0


In [23]:
from roms_tools import Grid, ROMSOutput

In [25]:
grid = Grid.from_file(model_grid_path)

In [29]:
roms_output_bgc = ROMSOutput(
    grid=grid,
    path=bgc_model_paths,
    use_dask=True,
)

roms_output_phys = ROMSOutput(
    grid=grid,
    path=phys_model_paths,
    use_dask=True,
)

In [31]:
# open grid
grid=xr.open_mfdataset(model_grid_path)

# regridding
h=roms_regrid(grid,grid['h'])

mask=roms_regrid(grid,grid['mask_rho'])

In [33]:
var_names=['ALK','DIC','PO4','NO3','SiO3','NH4','O2']
depth_levels=np.arange(0, 40,10)

In [ ]:
bgc = roms_output_bgc.regrid(depth_levels=depth_levels,var_names=var_names).load()
phys = roms_output_phys.regrid(depth_levels=depth_levels)

In [ ]:
time_index = phys.indexes['time']
unique_time = ~time_index.duplicated()

phys = phys.isel(time=unique_time)
phys = phys.sortby('time')

In [ ]:
time_index = bgc.indexes['time']
unique_time = ~time_index.duplicated()

bgc = bgc.isel(time=unique_time)
bgc = bgc.sortby('time')

In [ ]:
Sref=34
nTA_model=((bgc['ALK'].load()-TA0)/phys['salt'].thin({'time': 24}).load())*Sref+TA0

nDIC_model=((bgc['DIC'].load()-DIC0)/phys['salt'].thin({'time': 24}).load())*Sref+DIC0

bgc['nTA']=nTA_model
bgc['nDIC']=nDIC_model

In [ ]:
# Assuming locations is a list of lat/lon pairs
bgc_values = []

# Loop over the first 10 locations and store each selection in t_values
for i in range(7):
    lat, lon = locations[i]
    
    # Select the 't' values at the nearest lat/lon
    bgc_selected = bgc.sel(lat=lat, method='nearest').sel(lon=lon, method='nearest')
    
    # Store the result in the list
    bgc_values.append(bgc_selected)

# Combine the selections into an xarray Dataset or DataArray
bgc_values_combined = xr.concat(bgc_values, dim='location')

# Assign a location coordinate for clarity (optional)
bgc_values_combined = bgc_values_combined.assign_coords(location=('location', [1,3,5,7,9,10,12]))
bgc_values_combined['depth']=bgc_values_combined.depth*(-1)


In [ ]:
# define location which calculations the average location of each station
def get_location(obs, hv_values):
    locations = []
    for hv in hv_values:
        lat = (obs['lat'].sel(HV=hv).isel(depth_bin=0).mean('date').squeeze()).values+0
        lon = obs['lon'].sel(HV=hv).isel(depth_bin=0).mean('date').squeeze().values + 360
        locations.append([lat, lon])
    return locations

# List of HV values
hv_values = 1,3,5,7,9,10,12

# Get the locations
locations = get_location(obs, hv_values)

In [ ]:
# -----------------------------
# Extract time list from obs
# -----------------------------
time_list = np.unique(obs['time'].values)

# -----------------------------
# Select nearest model output
# -----------------------------
bgc_sub = bgc_values_combined.sel(
    time=time_list,
    method='nearest'
)

In [ ]:
bgc_sub.load()

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Define depth levels
levels = np.arange(0, 80)

# Create figure and subplots (3 rows: 1 for the map, 2 for boxplots)
fig = plt.figure(figsize=(12, 13))  # Wider figure for side-by-side plots
gs = fig.add_gridspec(3, 2, height_ratios=[2, 1, 1], width_ratios=[1, 1])  

# --- Shared Map (Top Row, Spanning Both Columns) ---
ax_map = fig.add_subplot(gs[0, :], projection=ccrs.PlateCarree())

# Plot bathymetry
cf1 = ax_map.contourf(h.lon, h.lat, h, levels, transform=ccrs.PlateCarree(), cmap='cividis')
ax_map.contourf(h.lon, h.lat, mask.where(mask != 1), cmap='Greys', transform=ccrs.PlateCarree())

# Add station labels
for i, e in enumerate(locations):
    ax_map.text(e[1], e[0], hv_values[i], color='red', size=10, ha='center', va='center', transform=ccrs.PlateCarree())

# Gridlines
gls = ax_map.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, color='darkgray', alpha=0.5, linestyle='--')
gls.top_labels = False
gls.right_labels = False

# Colorbar
cb1 = plt.colorbar(cf1, shrink=0.5, ax=ax_map, orientation='horizontal', pad=0.05)
cb1.set_label('Bottom Depth (m)', fontsize=10)

ax_map.set_extent([-22, -21.35, 64.25, 64.45], ccrs.PlateCarree())
ax_map.set_title('Station Overview')

# --- Function for Extracting Data ---
def extract_cruise_data(var, depth=None, num_cruises=10):
    """Extracts non-NaN values for a given variable across cruises."""
    cruises = []
    for i in range(num_cruises):
        data = np.hstack(var.isel(date=i) if depth is None else var.isel(date=i, depth_bin=depth))
        cruises.append(data[~np.isnan(data)])
    return cruises

def extract_model_data(var, depth=None, num_cruises=10):
    """Extracts non-NaN values from the model across time steps."""
    cruises = []
    for i in range(num_cruises):
        data = np.hstack(var.isel(time=i) if depth is None else var.isel(time=i, depth=depth))
        cruises.append(data[~np.isnan(data)])
    return cruises

# --- Boxplots for Observations (Left) and Model (Right) ---
ax_obs = fig.add_subplot(gs[1, 0])  # Observations
ax_model = fig.add_subplot(gs[1, 1])  # Model

ax_obs.boxplot(extract_cruise_data(nTA), patch_artist=True)
ax_obs.set_title('Salinity Normalized Alk Variation (Observed)')
ax_obs.set_ylabel('mmol/m$^3$')
ax_obs.set_xticklabels([])
ax_obs.set_ylim(2280,2370)

ax_model.boxplot(extract_model_data(bgc_sub['nTA']), patch_artist=True)
ax_model.set_title('Salinity Normalized Alk Variation (Model)')
ax_model.set_xticklabels([])
ax_model.set_ylim(2230,2320)

# --- Surface-Only Boxplots for Observations (Left) and Model (Right)---
ax_obs_surface = fig.add_subplot(gs[2, 0])  # Observations (Surface)
ax_model_surface = fig.add_subplot(gs[2, 1])  # Model (Surface)

ax_obs_surface.boxplot(extract_cruise_data(nTA, depth=0), patch_artist=True)
ax_obs_surface.set_title('Salinity Normalized Alk, Surface Only (Observed)')
ax_obs_surface.set_xlabel('Cruise')
ax_obs_surface.set_ylabel('mmol/m$^3$')
ax_obs_surface.set_xticklabels(pd.to_datetime(obs['date'].isel(date=slice(0,10))).strftime('%d/%m/%Y'), rotation=45)
ax_obs_surface.set_ylim(2310,2390)

ax_model_surface.boxplot(extract_model_data(bgc_sub['nTA'], depth=0), patch_artist=True)
ax_model_surface.set_title('Salinity Normalized Alk, Surface Only (Model)')
ax_model_surface.set_xlabel('Cruise')
ax_model_surface.set_ylabel('mmol/m$^3$')
ax_model_surface.set_xticklabels(pd.to_datetime(bgc_sub['time']).strftime('%d/%m/%Y'), rotation=45)
ax_model_surface.set_ylim(2230,2310)

plt.tight_layout()
plt.show()


In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Define depth levels
levels = np.arange(0, 80)

# Create figure and subplots (3 rows: 1 for the map, 2 for boxplots)
fig = plt.figure(figsize=(12, 7))  # Wider figure for side-by-side plots
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], width_ratios=[1, 1])  



# --- Function for Extracting Data ---
def extract_cruise_data(var, depth=None, num_cruises=10):
    """Extracts non-NaN values for a given variable across cruises."""
    cruises = []
    for i in range(num_cruises):
        data = np.hstack(var.isel(date=i) if depth is None else var.isel(date=i, depth_bin=depth))
        cruises.append(data[~np.isnan(data)])
    return cruises

def extract_model_data(var, depth=None, num_cruises=10):
    """Extracts non-NaN values from the model across time steps."""
    cruises = []
    for i in range(num_cruises):
        data = np.hstack(var.isel(time=i) if depth is None else var.isel(time=i, depth=depth))
        cruises.append(data[~np.isnan(data)])
    return cruises

# --- Boxplots for Observations (Left) and Model (Right) ---
ax_obs = fig.add_subplot(gs[0, 0])  # Observations
ax_model = fig.add_subplot(gs[0, 1])  # Model

ax_obs.boxplot(extract_cruise_data(nDIC), patch_artist=True)
ax_obs.set_title('Salinity Normalized DIC Variation (Observed)')
ax_obs.set_ylabel('mmol/m$^3$')
ax_obs.set_xticklabels([])
ax_obs.set_ylim(2070,2180)

ax_model.boxplot(extract_model_data(bgc_sub['nDIC']), patch_artist=True)
ax_model.set_title('Salinity Normalized DIC Variation (Model)')
ax_model.set_xticklabels([])
ax_model.set_ylim(2030,2140)

# --- Surface-Only Boxplots for Observations (Left) and Model (Right) ---
ax_obs_surface = fig.add_subplot(gs[1, 0])  # Observations (Surface)
ax_model_surface = fig.add_subplot(gs[1, 1])  # Model (Surface)

ax_obs_surface.boxplot(extract_cruise_data(nDIC, depth=0), patch_artist=True)
ax_obs_surface.set_title('Salinity Normalized DIC, Surface Only (Observed)')
ax_obs_surface.set_xlabel('Cruise')
ax_obs_surface.set_ylabel('mmol/m$^3$')
ax_obs_surface.set_xticklabels(pd.to_datetime(obs['date'].isel(date=slice(0,10))).strftime('%d/%m/%Y'), rotation=45)
ax_obs_surface.set_ylim(2070,2180)

ax_model_surface.boxplot(extract_model_data(bgc_sub['nDIC'], depth=0), patch_artist=True)
ax_model_surface.set_title('Salinity Normalized DIC, Surface Only (Model)')
ax_model_surface.set_xlabel('Cruise')
ax_model_surface.set_ylabel('mmol/m$^3$')
ax_model_surface.set_xticklabels(pd.to_datetime(bgc_sub['time'].isel(time=slice(0,10))).strftime('%d/%m/%Y'), rotation=45)
ax_model_surface.set_ylim()
ax_model_surface.set_ylim(2030,2150)

plt.tight_layout()
plt.show()


In [ ]:
loc = obs['lon'].sel(HV=[1,5,9,10]).isel(depth_bin=0).mean('date').values
depth=obs['depth_bin'].values
levels=np.arange(2290,2360,2)

bins = np.arange(0, 90, 10) 
#depth_bins = xr.DataArray(pd.cut(depth, bins, labels=bins[:-1]), dims="depth", name="binned_depth")

data_alk1=nTA.sel(HV=[1,5,9,10]).isel(date=slice(0,10)).mean('date')
data_alk2=obs['Alk'].sel(HV=[1,5,9,10]).isel(date=slice(0,10)).mean('date')

fig, axarr = plt.subplots(nrows=1, ncols=2, figsize=(6*2, 3), constrained_layout=True)
ax = axarr.flatten()

#c0=ax[0].contourf(loc, data_alk.depth, data_alk.transpose())
c0=ax[0].contourf(loc,data_alk1.depth_bin,data_alk1.transpose(),levels)
plt.colorbar(c0,ax=ax[0], orientation='vertical', label='mmol/m3',shrink=0.5)
ax[0].invert_yaxis()
ax[0].set_title('salinity normalized alkalinity along fjord (observed)')
ax[0].set_xlabel('longitude')
ax[0].set_ylabel('depth (m)')

c0=ax[1].contourf(loc,data_alk2.depth_bin,data_alk2.transpose(),levels)
plt.colorbar(c0,ax=ax[1], orientation='vertical', label='mmol/m3',shrink=0.5)
ax[1].invert_yaxis()
ax[1].set_title('alkalinity along fjord (observed)')
ax[1].set_xlabel('longitude')
ax[1].set_ylabel('depth (m)')

In [ ]:
loc = obs['lon'].sel(HV=[1,5,9,10]).isel(depth_bin=0).mean('date').values
depth=obs['depth_bin'].values
levels=np.arange(2230,2300,2)

bins = np.arange(0, 90, 10) 
#depth_bins = xr.DataArray(pd.cut(depth, bins, labels=bins[:-1]), dims="depth", name="binned_depth")

data_alk1=bgc_sub['nTA'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).mean('time')
data_alk2=bgc_sub['ALK'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).mean('time')

fig, axarr = plt.subplots(nrows=1, ncols=2, figsize=(6*2, 3), constrained_layout=True)
ax = axarr.flatten()

#c0=ax[0].contourf(loc, data_alk.depth, data_alk.transpose())
c0=ax[0].contourf(loc,data_alk1.depth*(-1),data_alk1.transpose(),levels)
plt.colorbar(c0,ax=ax[0], orientation='vertical', label='mmol/m3',shrink=0.5)
ax[0].set_ylim(0,40)
ax[0].invert_yaxis()
ax[0].set_title('salinity normalized alkalinity along fjord (modeled)')
ax[0].set_xlabel('longitude')
ax[0].set_ylabel('depth (m)')


c0=ax[1].contourf(loc,data_alk2.depth*(-1),data_alk2.transpose(),levels)
plt.colorbar(c0,ax=ax[1], orientation='vertical', label='mmol/m3',shrink=0.5)
ax[1].set_ylim(0,40)
ax[1].invert_yaxis()
ax[1].set_title('alkalinity along fjord (modeled)')
ax[1].set_xlabel('longitude')
ax[1].set_ylabel('depth (m)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute mean and standard deviation over time for observations
depth_obs = obs['depth_bin'].values
depth_model = bgc_sub['depth'].values

# Observations (Mean and Std Dev)
alk_mean_obs = obs['DIC'].sel(HV=[1,5,9,10]).isel(date=slice(0,10)).mean(dim=['date', 'HV'])
alk_std_obs = obs['DIC'].sel(HV=[1,5,9,10]).isel(date=slice(0,10)).std(dim=['date', 'HV'])

nalk_mean_obs = nDIC.sel(HV=[1,5,9,10]).isel(date=slice(0,10)).mean(dim=['date', 'HV'])
nalk_std_obs = nDIC.sel(HV=[1,5,9,10]).isel(date=slice(0,10)).std(dim=['date', 'HV'])

# Model (Mean and Std Dev)
alk_mean_model = bgc_sub['DIC'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).mean(dim=['time', 'location'])
alk_std_model = bgc_sub['DIC'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).std(dim=['time', 'location'])

nalk_mean_model = bgc_sub['nDIC'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).mean(dim=['time', 'location'])
nalk_std_model = bgc_sub['nDIC'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).std(dim=['time', 'location'])

# Create figure with 2 columns (nTA on left, TA on right)
fig, axarr = plt.subplots(nrows=1, ncols=2, figsize=(12, 6), constrained_layout=True)
ax = axarr.flatten()

# --- Plot Salinity Normalized DIC (nTA) ---
ax[0].plot(
    nalk_mean_obs, depth_obs,
    marker='o', linestyle='-',
    color='orange', label='Observed DIC'
)
ax[0].fill_betweenx(
    depth_obs,
    nalk_mean_obs - nalk_std_obs,
    nalk_mean_obs + nalk_std_obs,
    color='orange', alpha=0.3
)

ax[0].plot(
    nalk_mean_model, depth_model * (-1),
    marker='s', linestyle='--',
    color='black', label='Modeled DIC'
)
ax[0].fill_betweenx(
    depth_model * (-1),
    nalk_mean_model - nalk_std_model,
    nalk_mean_model + nalk_std_model,
    color='grey', alpha=0.3
)

ax[0].set_ylim(0, 30)
ax[0].invert_yaxis()
ax[0].set_xlabel('Salinity Normalized DIC (mmol/m³)')
ax[0].set_ylabel('Depth (m)')
ax[0].set_title('Salinity Normalized DIC Profile')
ax[0].legend()
ax[0].grid()


# --- Plot Total DIC ---
ax[1].plot(
    alk_mean_obs, depth_obs,
    marker='o', linestyle='-',
    color='orange', label='Observed DIC'
)
ax[1].fill_betweenx(
    depth_obs,
    alk_mean_obs - alk_std_obs,
    alk_mean_obs + alk_std_obs,
    color='orange', alpha=0.3
)

ax[1].plot(
    alk_mean_model, depth_model * (-1),
    marker='s', linestyle='--',
    color='black', label='Modeled DIC'
)
ax[1].fill_betweenx(
    depth_model * (-1),
    alk_mean_model - alk_std_model,
    alk_mean_model + alk_std_model,
    color='grey', alpha=0.3
)

ax[1].set_ylim(0, 30)
ax[1].invert_yaxis()
ax[1].set_xlabel('Total DIC (mmol/m³)')
ax[1].set_title('Total DIC Profile')
ax[1].legend()
ax[1].grid()

# Label c)
ax[0].text(-0.05, 1.05, 'c)', transform=ax[0].transAxes, 
           fontsize=16, fontweight='bold', va='top', ha='right')
# Label d)
ax[1].text(-0.05, 1.05, 'd)', transform=ax[1].transAxes, 
           fontsize=16, fontweight='bold', va='top', ha='right')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Compute mean and standard deviation over time for observations
depth_obs = obs['depth_bin'].values
depth_model = bgc_sub['depth'].values

# Observations (Mean and Std Dev)
alk_mean_obs = obs['Alk'].sel(HV=[1,5,9,10]).isel(date=slice(0,10)).mean(dim=['date', 'HV'])
alk_std_obs = obs['Alk'].sel(HV=[1,5,9,10]).isel(date=slice(0,10)).std(dim=['date', 'HV'])

nalk_mean_obs = nTA.sel(HV=[1,5,9,10]).isel(date=slice(0,10)).mean(dim=['date', 'HV'])
nalk_std_obs = nTA.sel(HV=[1,5,9,10]).isel(date=slice(0,10)).std(dim=['date', 'HV'])

# Model (Mean and Std Dev)
alk_mean_model = bgc_sub['ALK'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).mean(dim=['time', 'location'])
alk_std_model = bgc_sub['ALK'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).std(dim=['time', 'location'])

nalk_mean_model = bgc_sub['nTA'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).mean(dim=['time', 'location'])
nalk_std_model = bgc_sub['nTA'].sel(location=[1,5,9,10]).isel(time=slice(2,7)).std(dim=['time', 'location'])

# Create figure with 2 columns (nTA on left, TA on right)
fig, axarr = plt.subplots(nrows=1, ncols=2, figsize=(12, 6), constrained_layout=True)
ax = axarr.flatten()

# --- Plot Salinity Normalized DIC (nTA) ---
ax[0].plot(
    nalk_mean_obs, depth_obs,
    marker='o', linestyle='-',
    color='orange', label='Observed nTA'
)
ax[0].fill_betweenx(
    depth_obs,
    nalk_mean_obs - nalk_std_obs,
    nalk_mean_obs + nalk_std_obs,
    color='orange', alpha=0.3
)

ax[0].plot(
    nalk_mean_model, depth_model * (-1),
    marker='s', linestyle='--',
    color='black', label='Modeled nTA'
)
ax[0].fill_betweenx(
    depth_model * (-1),
    nalk_mean_model - nalk_std_model,
    nalk_mean_model + nalk_std_model,
    color='grey', alpha=0.3
)

ax[0].set_ylim(0, 30)
ax[0].invert_yaxis()
ax[0].set_xlabel('Salinity Normalized Alk (mmol/m³)')
ax[0].set_ylabel('Depth (m)')
ax[0].set_title('Salinity Normalized Alk Profile')
ax[0].legend()
ax[0].grid()


# --- Plot Total DIC ---
ax[1].plot(
    alk_mean_obs, depth_obs,
    marker='o', linestyle='-',
    color='orange', label='Observed Alk'
)
ax[1].fill_betweenx(
    depth_obs,
    alk_mean_obs - alk_std_obs,
    alk_mean_obs + alk_std_obs,
    color='orange', alpha=0.3
)

ax[1].plot(
    alk_mean_model, depth_model * (-1),
    marker='s', linestyle='--',
    color='black', label='Modeled Alk'
)
ax[1].fill_betweenx(
    depth_model * (-1),
    alk_mean_model - alk_std_model,
    alk_mean_model + alk_std_model,
    color='grey', alpha=0.3
)

ax[1].set_ylim(0, 30)
ax[1].invert_yaxis()
ax[1].set_xlabel('Total ALK (mmol/m³)')
ax[1].set_title('Total ALK Profile')
ax[1].legend()
ax[1].grid()

# Label a)
ax[0].text(-0.05, 1.05, 'a)', transform=ax[0].transAxes, 
           fontsize=16, fontweight='bold', va='top', ha='right')
# Label b)
ax[1].text(-0.05, 1.05, 'b)', transform=ax[1].transAxes, 
           fontsize=16, fontweight='bold', va='top', ha='right')

plt.show()


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# --------------------------------------------------
# Open dataset
# --------------------------------------------------
ds = xr.open_dataset(
    "/anvil/projects/x-ees250129/Datasets/UNIFIED/BGCdataset.nc"
)

# --------------------------------------------------
# Select surface layer
# --------------------------------------------------
if "dep" in ds.dims:
    Alk_surf = ds["Alk"].isel(dep=0)
    DIC_surf = ds["DIC"].isel(dep=0)
elif "z" in ds.dims:
    Alk_surf = ds["Alk"].isel(z=0)
    DIC_surf = ds["DIC"].isel(z=0)
else:
    raise ValueError("No recognized vertical dimension found.")

# --------------------------------------------------
# Coordinates
# --------------------------------------------------
lat = ds["lat"]
lon = ds["lon"]

# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, axs = plt.subplots(
    ncols=2,
    figsize=(12, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
    dpi=300
)

# ---- ALK ----
cf1 = axs[0].contourf(
    lon, lat, Alk_surf[0, :, :],
    levels=np.arange(2280,2360,5),
    cmap="viridis",
    transform=ccrs.PlateCarree()
)
axs[0].set_title("Surface Alkalinity")
plt.colorbar(cf1, ax=axs[0], orientation="vertical", label="ALK")

# ---- DIC ----
cf2 = axs[1].contourf(
    lon, lat, DIC_surf[6, :, :],
    levels=np.arange(1900,2100,10),
    cmap="plasma",
    transform=ccrs.PlateCarree()
)
axs[1].set_title("Surface DIC")
plt.colorbar(cf2, ax=axs[1], orientation="vertical", label="DIC")

# --------------------------------------------------
# Zoom to Iceland
# --------------------------------------------------
# (SW Iceland–centered, but includes whole island)
lon_min, lon_max = -32, -10
lat_min, lat_max = 60, 68

for ax in axs:
    ax.set_extent([lon_min, lon_max, lat_min, lat_max],
                  crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m")
    ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
loc = obs['lon'].sel(HV=[1,5,9,10]).isel(depth_bin=0).mean('date').values
depth=obs['depth_bin'].values
levels=np.arange(2070,2160,2)

bins = np.arange(0, 90, 10) 
#depth_bins = xr.DataArray(pd.cut(depth, bins, labels=bins[:-1]), dims="depth", name="binned_depth")

data_alk1=nDIC.sel(HV=[1,5,9,10]).isel(date=slice(0,4)).mean('date')
data_alk2=nDIC.sel(HV=[1,5,9,10]).isel(date=slice(5,10)).mean('date')

fig, axarr = plt.subplots(nrows=1, ncols=2, figsize=(6*2, 3), constrained_layout=True)
ax = axarr.flatten()

#c0=ax[0].contourf(loc, data_alk.depth, data_alk.transpose())
c0=ax[0].contourf(loc,data_alk1.depth_bin,data_alk1.transpose(),levels)
plt.colorbar(c0,ax=ax[0], orientation='vertical', label='mmol/m3',shrink=0.5)
ax[0].invert_yaxis()
ax[0].set_title('salinity normalized DIC along fjord April-May (observed)')
ax[0].set_xlabel('longitude')
ax[0].set_ylabel('depth (m)')

c0=ax[1].contourf(loc,data_alk2.depth_bin,data_alk2.transpose(),levels)
plt.colorbar(c0,ax=ax[1], orientation='vertical', label='mmol/m3',shrink=0.5)
ax[1].invert_yaxis()
ax[1].set_title('salinity normalized DIC along fjord June-August (observed)')
ax[1].set_xlabel('longitude')
ax[1].set_ylabel('depth (m)')

In [ ]:
loc = obs['lon'].sel(HV=[1,5,9,10]).isel(depth_bin=0).mean('date').values
depth=obs['depth_bin'].values
levels=np.arange(2070,2160,2)

bins = np.arange(0, 90, 10) 
#depth_bins = xr.DataArray(pd.cut(depth, bins, labels=bins[:-1]), dims="depth", name="binned_depth")

data_alk1=nDIC.sel(HV=[1,5,9,10]).isel(date=slice(0,4)).mean('date')
data_alk2=nDIC.sel(HV=[1,5,9,10]).isel(date=slice(5,10)).mean('date')

data_alk1=bgc_sub['nDIC'].sel(location=[1,5,9,10]).isel(time=slice(0,4)).mean('time')
data_alk2=bgc_sub['nDIC'].sel(location=[1,5,9,10]).isel(time=slice(5,10)).mean('time')


fig, axarr = plt.subplots(nrows=1, ncols=2, figsize=(6*2, 3), constrained_layout=True)
ax = axarr.flatten()

#c0=ax[0].contourf(loc, data_alk.depth, data_alk.transpose())
c0=ax[0].contourf(loc,data_alk1.depth*(-1),data_alk1.transpose(),levels)
plt.colorbar(c0,ax=ax[0], orientation='vertical', label='mmol/m3',shrink=0.5)
ax[0].set_ylim(0,40)
ax[0].invert_yaxis()
ax[0].set_title('salinity normalized DIC along fjord April-May (modeled)')
ax[0].set_xlabel('longitude')
ax[0].set_ylabel('depth (m)')

c0=ax[1].contourf(loc,data_alk2.depth*(-1),data_alk2.transpose(),levels)
plt.colorbar(c0,ax=ax[1], orientation='vertical', label='mmol/m3',shrink=0.5)
ax[1].set_ylim(0,40)
ax[1].invert_yaxis()
ax[1].set_title('salinity normalized DIC along fjord June-August (modeled )')
ax[1].set_xlabel('longitude')
ax[1].set_ylabel('depth (m)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Create figure and subplots (2x2)
fig, axes = plt.subplots(
    nrows=2, ncols=2, figsize=(14, 8), sharex=False
)

# --- Data Extraction Functions ---
def extract_depth_avg(var, depth_range):
    """Extract mean across depth range for each time step (observations)."""
    return var.sel(depth_bin=slice(*depth_range)).mean(dim='depth_bin', skipna=True)

def extract_depth_avg_model(var, depth_range):
    """Extract mean across depth range for each time step (model)."""
    return var.sel(depth=slice(*depth_range)).mean(dim='depth', skipna=True)

# --- Depth ranges ---
upper_depth_range = (0, 5)
lower_depth_range = (20, 40)

upper_depth_range_m = (-0.5, -5)
lower_depth_range_m = (-20, -40)

# =======================
# TOP ROW — nDIC
# =======================

# --- Observed nDIC ---
ax = axes[0, 0]
obs_upper = extract_depth_avg(nDIC.mean('HV'), upper_depth_range)
obs_lower = extract_depth_avg(nDIC.mean('HV'), lower_depth_range)

ax.plot(obs['date'], obs_upper, label='0–10 m', marker='o')
ax.plot(obs['date'], obs_lower, label='20–40 m', marker='s', linestyle='--')
ax.set_title('Salinity-Normalized DIC (Observed)')
ax.set_ylabel('mmol m$^{-3}$')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# --- Modeled nDIC ---
ax = axes[0, 1]
model_upper = extract_depth_avg_model(
    bgc_sub['nDIC'].mean('location'), upper_depth_range_m
)
model_lower = extract_depth_avg_model(
    bgc_sub['nDIC'].mean('location'), lower_depth_range_m
)

ax.plot(bgc_sub['time'], model_upper, label='0–10 m', marker='o')
ax.plot(bgc_sub['time'], model_lower, label='20–40 m', marker='s', linestyle='--')
ax.set_title('Salinity-Normalized DIC (Model)')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# =======================
# BOTTOM ROW — ALK
# =======================

# --- Observed ALK ---
ax = axes[1, 0]
obs_upper = extract_depth_avg(nTA.mean('HV'), upper_depth_range)
obs_lower = extract_depth_avg(nTA.mean('HV'), lower_depth_range)

ax.plot(obs['date'], obs_upper, label='0–10 m', marker='o')
ax.plot(obs['date'], obs_lower, label='20–40 m', marker='s', linestyle='--')
ax.set_title('Alkalinity (Observed)')
ax.set_ylabel('mmol m$^{-3}$')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# --- Modeled ALK ---
ax = axes[1, 1]
model_upper = extract_depth_avg_model(
    bgc_sub['nTA'].mean('location'), upper_depth_range_m
)
model_lower = extract_depth_avg_model(
    bgc_sub['nTA'].mean('location'), lower_depth_range_m
)

ax.plot(bgc_sub['time'], model_upper, label='0–10 m', marker='o')
ax.plot(bgc_sub['time'], model_lower, label='20–40 m', marker='s', linestyle='--')
ax.set_title('Alkalinity (Model)')
ax.legend()
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Create figure and subplots (4 rows for nutrients, 2 columns for observations vs. model)
fig = plt.figure(figsize=(12, 13))  # Wider figure for side-by-side comparison
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 1], width_ratios=[1, 1])  

# --- Data Extraction Functions ---
def extract_cruise_data(var, depth=0, num_cruises=10):
    """Extracts non-NaN values for a given variable across cruises (observations)."""
    cruises = []
    for i in range(num_cruises):
        data = np.hstack(var.isel(date=i) if depth is None else var.isel(date=i, depth_bin=depth))
        cruises.append(data[~np.isnan(data)])
    return cruises

def extract_model_data(var, depth=0, num_cruises=10):
    """Extracts non-NaN values from the model across time steps."""
    cruises = []
    for i in range(num_cruises):
        data = np.hstack(var.isel(time=i) if depth is None else var.isel(time=i, depth=depth))
        cruises.append(data[~np.isnan(data)])
    return cruises

# --- NO3 Boxplots ---
ax1_obs = fig.add_subplot(gs[0, 0])  # Observations
ax1_model = fig.add_subplot(gs[0, 1])  # Model

ax1_obs.boxplot(extract_cruise_data(obs['NO3']), patch_artist=True)
ax1_obs.set_title('NO₃ Surface Only (Observed)')
ax1_obs.set_ylabel('mmol/m$^3$')
ax1_obs.set_xticklabels([])

ax1_model.boxplot(extract_model_data(bgc_sub['NO3']), patch_artist=True)
ax1_model.set_title('NO₃ Surface Only (Model)')
ax1_model.set_xticklabels([])

# --- PO4 Boxplots ---
ax2_obs = fig.add_subplot(gs[1, 0])  # Observations
ax2_model = fig.add_subplot(gs[1, 1])  # Model

ax2_obs.boxplot(extract_cruise_data(obs['PO4']), patch_artist=True)
ax2_obs.set_title('PO₄ Surface Only (Observed)')
ax2_obs.set_ylabel('mmol/m$^3$')
ax2_obs.set_xticklabels([])

ax2_model.boxplot(extract_model_data(bgc_sub['PO4']), patch_artist=True)
ax2_model.set_title('PO₄ Surface Only (Model)')
ax2_model.set_xticklabels([])

# --- SiO2 Boxplots ---
ax3_obs = fig.add_subplot(gs[2, 0])  # Observations
ax3_model = fig.add_subplot(gs[2, 1])  # Model

ax3_obs.boxplot(extract_cruise_data(obs['SiO2']), patch_artist=True)
ax3_obs.set_title('SiO₂ Surface Only (Observed)')
ax3_obs.set_ylabel('mmol/m$^3$')
ax3_obs.set_xticklabels([])

ax3_model.boxplot(extract_model_data(bgc_sub['SiO3']), patch_artist=True)
ax3_model.set_title('SiO₂ Surface Only (Model)')
ax3_model.set_xticklabels([])

# --- NH4 Boxplots ---
ax4_obs = fig.add_subplot(gs[3, 0])  # Observations
ax4_model = fig.add_subplot(gs[3, 1])  # Model

ax4_obs.boxplot(extract_cruise_data(obs['NH4']), patch_artist=True)
ax4_obs.set_title('NH₄ Surface Only (Observed)')
ax4_obs.set_xlabel('Cruise')
ax4_obs.set_ylabel('mmol/m$^3$')
ax4_obs.set_xticklabels(pd.to_datetime(obs['date'].isel(date=slice(0,10))).strftime('%d/%m/%Y'), rotation=45)

ax4_model.boxplot(extract_model_data(bgc_sub['NH4']), patch_artist=True)
ax4_model.set_title('NH₄ Surface Only (Model)')
ax4_model.set_xlabel('Cruise')
ax4_model.set_ylabel('mmol/m$^3$')
ax4_model.set_xticklabels(pd.to_datetime(bgc_sub['time'].isel(time=slice(0,10))).strftime('%d/%m/%Y'), rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Create figure and subplots (4 rows for nutrients, 2 columns for observations vs. model)
fig = plt.figure(figsize=(12, 13))  # Wider figure for side-by-side comparison
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 1], width_ratios=[1, 1])  

# --- Data Extraction Functions ---
def extract_cruise_data(var, num_cruises=10):
    """Extracts non-NaN values for a given variable across all depths and cruises (observations)."""
    cruises = []
    for i in range(num_cruises):
        data = np.hstack(var.isel(date=i))  # Include all depth levels
        cruises.append(data[~np.isnan(data)])
    return cruises

def extract_model_data(var, num_cruises=10):
    """Extracts non-NaN values for a given variable across all depths and time steps (model)."""
    cruises = []
    for i in range(num_cruises):
        data = np.hstack(var.isel(time=i))  # Include all depth levels
        cruises.append(data[~np.isnan(data)])
    return cruises

# --- NO3 Boxplots (All Depths) ---
ax1_obs = fig.add_subplot(gs[0, 0])  # Observations
ax1_model = fig.add_subplot(gs[0, 1])  # Model

ax1_obs.boxplot(extract_cruise_data(obs['NO3']), patch_artist=True)
ax1_obs.set_title('NO₃ (All Depths, Observed)')
ax1_obs.set_ylabel('mmol/m$^3$')
ax1_obs.set_xticklabels([])

ax1_model.boxplot(extract_model_data(bgc_sub['NO3']), patch_artist=True)
ax1_model.set_title('NO₃ (All Depths, Model)')
ax1_model.set_xticklabels([])

# --- PO4 Boxplots (All Depths) ---
ax2_obs = fig.add_subplot(gs[1, 0])  # Observations
ax2_model = fig.add_subplot(gs[1, 1])  # Model

ax2_obs.boxplot(extract_cruise_data(obs['PO4']), patch_artist=True)
ax2_obs.set_title('PO₄ (All Depths, Observed)')
ax2_obs.set_ylabel('mmol/m$^3$')
ax2_obs.set_xticklabels([])

ax2_model.boxplot(extract_model_data(bgc_sub['PO4']), patch_artist=True)
ax2_model.set_title('PO₄ (All Depths, Model)')
ax2_model.set_xticklabels([])

# --- SiO2 Boxplots (All Depths) ---
ax3_obs = fig.add_subplot(gs[2, 0])  # Observations
ax3_model = fig.add_subplot(gs[2, 1])  # Model

ax3_obs.boxplot(extract_cruise_data(obs['SiO2']), patch_artist=True)
ax3_obs.set_title('SiO₂ (All Depths, Observed)')
ax3_obs.set_ylabel('mmol/m$^3$')
ax3_obs.set_xticklabels([])

ax3_model.boxplot(extract_model_data(bgc_sub['SiO3']), patch_artist=True)
ax3_model.set_title('SiO₂ (All Depths, Model)')
ax3_model.set_xticklabels([])

# --- NH4 Boxplots (All Depths) ---
ax4_obs = fig.add_subplot(gs[3, 0])  # Observations
ax4_model = fig.add_subplot(gs[3, 1])  # Model

ax4_obs.boxplot(extract_cruise_data(obs['NH4']), patch_artist=True)
ax4_obs.set_title('NH₄ (All Depths, Observed)')
ax4_obs.set_xlabel('Cruise')
ax4_obs.set_ylabel('mmol/m$^3$')
ax4_obs.set_xticklabels(pd.to_datetime(obs['date'].isel(date=slice(0,10))).strftime('%d/%m/%Y'), rotation=45)

ax4_model.boxplot(extract_model_data(bgc_sub['NH4']), patch_artist=True)
ax4_model.set_title('NH₄ (All Depths, Model)')
ax4_model.set_xlabel('Cruise')
ax4_model.set_ylabel('mmol/m$^3$')
ax4_model.set_xticklabels(pd.to_datetime(bgc_sub['time'].isel(time=slice(0,10))).strftime('%d/%m/%Y'), rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Create figure and subplots (4 rows for nutrients, 2 columns for observations vs. model)
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(12, 13), sharex=False)  

# --- Data Extraction Functions ---
def extract_depth_avg(var, depth_range, time_dim='date'):
    """Extracts mean value across a given depth range for each time step."""
    return var.sel(depth_bin=slice(*depth_range)).mean(dim='depth_bin', skipna=True)

def extract_depth_avg_model(var, depth_range, time_dim='time'):
    """Extracts mean value across a given depth range for each time step."""
    return var.sel(depth=slice(*depth_range)).mean(dim='depth', skipna=True)

# Define depth ranges
upper_depth_range = (0,5)   # Upper level (0-10 m)
lower_depth_range = (20, 40)  # Lower level (20-40 m)

upper_depth_range_m = (-0.5,-5)   # Upper level (0-10 m)
lower_depth_range_m = (-20, -40)  # Lower level (20-40 m)

# Define nutrients and their respective labels
nutrients = ['NO3', 'PO4', 'SiO2', 'NH4']
nutrients_m = ['NO3', 'PO4', 'SiO3', 'NH4']
titles = ['NO₃', 'PO₄', 'SiO$_3$', 'NH₄']

# --- Plot each nutrient ---
for i, nutrient in enumerate(nutrients):
    # Observations
    ax_obs = axes[i, 0]
    obs_upper = extract_depth_avg(obs[nutrient].mean('HV'), upper_depth_range)
    obs_lower = extract_depth_avg(obs[nutrient].mean('HV'), lower_depth_range)

    ax_obs.plot(obs['date'], obs_upper, label='0-10m', marker='o', linestyle='-')
    ax_obs.plot(obs['date'], obs_lower, label='20-40m', marker='s', linestyle='--')
    ax_obs.set_title(f'{titles[i]} (Observed)')
    ax_obs.set_ylabel('mmol/m$^3$')
    ax_obs.legend()
    ax_obs.tick_params(axis='x', rotation=45)  # Rotate x-axis labels


for i, nutrient in enumerate(nutrients_m):
    # Model
    ax_model = axes[i, 1]
    model_upper = extract_depth_avg_model(bgc_sub[nutrient].mean('location'), upper_depth_range_m, time_dim='time')
    model_lower = extract_depth_avg_model(bgc_sub[nutrient].mean('location'), lower_depth_range_m, time_dim='time')

    ax_model.plot(bgc_sub['time'], model_upper, label='0-10m', marker='o', linestyle='-')
    ax_model.plot(bgc_sub['time'], model_lower, label='20-40m', marker='s', linestyle='--')
    ax_model.set_title(f'{titles[i]} (Model)')
    ax_model.legend()
    ax_model.tick_params(axis='x', rotation=45)  # Rotate x-axis labels


plt.tight_layout()
plt.show()
